## **[1]**

In [3]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, LogisticRegression, TweedieRegressor

In [4]:
def partition_by_key(data, xs, ys, split=0.1):
    D = {}
    for i in range(len(data)):
        k = tuple([])
        for x in xs: k += (data.loc[i][x],)
        for y in ys: k += (data.loc[i][y],)
        if k in D: D[k] += [i]
        else:      D[k]  = [i]
    ks      = list(D)
    train_i = list(np.random.choice(range(len(ks)), int(len(ks)*(1.0-split)), replace=False))
    test_i  = list(set(range(len(ks))).difference(set(train_i)))

    train, test = [], []
    for i in range(len(ks)):
        if i in train_i: train += D[ks[i]]
        else:            test  += D[ks[i]]
    return train, test

def mean_power_error(y_true, y_pred, power=1):
    diff = np.sum(np.abs(y_true - y_pred**power), axis=0) / len(y_true)
    return diff

In [5]:
data = pd.read_csv("NHANES.csv")

data.shape          # rows, cols
data.head(10)       # first 10
data.iloc[0:5, 0:8] # slice rows/cols
data.columns

Index(['SurveyYr', 'ID', 'Gender', 'Age', 'AgeDecade', 'AgeMonths', 'Race1',
       'Race3', 'Education', 'MaritalStatus', 'HHIncome', 'HHIncomeMid',
       'Poverty', 'HomeRooms', 'HomeOwn', 'Work', 'Weight', 'Length',
       'HeadCirc', 'Height', 'BMI', 'BMICatUnder20yrs', 'BMI_WHO', 'Pulse',
       'BPSysAve', 'BPDiaAve', 'BPSys1', 'BPDia1', 'BPSys2', 'BPDia2',
       'BPSys3', 'BPDia3', 'Testosterone', 'DirectChol', 'TotChol',
       'UrineVol1', 'UrineFlow1', 'UrineVol2', 'UrineFlow2', 'Diabetes',
       'DiabetesAge', 'HealthGen', 'DaysPhysHlthBad', 'DaysMentHlthBad',
       'LittleInterest', 'Depressed', 'nPregnancies', 'nBabies', 'Age1stBaby',
       'SleepHrsNight', 'SleepTrouble', 'PhysActive', 'PhysActiveDays',
       'TVHrsDay', 'CompHrsDay', 'TVHrsDayChild', 'CompHrsDayChild',
       'Alcohol12PlusYr', 'AlcoholDay', 'AlcoholYear', 'SmokeNow', 'Smoke100',
       'Smoke100n', 'SmokeAge', 'Marijuana', 'AgeFirstMarij', 'RegularMarij',
       'AgeRegMarij', 'HardDrugs', 'SexEve

In [6]:
data.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SurveyYr,10000,2,2009_10,5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ID,10000.0,NaN,NaN,NaN,61944.6438,5871.16716,51624.0,56904.5,62159.5,67039.0,71915.0
Gender,10000,2,female,5020,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,10000.0,NaN,NaN,NaN,36.7421,22.397566,0.0,17.0,36.0,54.0,80.0
AgeDecade,9667,8,40-49,1398,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
SexAge,5540.0,NaN,NaN,NaN,17.4287,3.716551,9.0,15.0,17.0,19.0,50.0
SexNumPartnLife,5725.0,NaN,NaN,NaN,15.085066,57.846434,0.0,2.0,5.0,12.0,2000.0
SexNumPartYear,4928.0,NaN,NaN,NaN,1.34233,2.782688,0.0,1.0,1.0,1.0,69.0
SameSex,5768,2,No,5353,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
def is_nonneg_int_series(s):
    s = s.dropna()
    if len(s) == 0: 
        return False
    # check "integer-like"
    return np.all(np.isclose(s, np.round(s))) and (s.min() >= 0)

binary_cols = []
numeric_cols = []
count_cols = []

for c in data.columns:
    s = data[c]
    nun = s.dropna().nunique()

    # object/category columns -> treat as categorical
    if s.dtype == "object":
        if nun == 2: binary_cols.append(c)
        continue

    # numeric columns
    if nun == 2:
        binary_cols.append(c)
    else:
        numeric_cols.append(c)
        if is_nonneg_int_series(s) and s.dropna().max() > 5:
            count_cols.append(c)

binary_cols, count_cols, numeric_cols[:10]

(['SurveyYr',
  'Gender',
  'Diabetes',
  'LittleInterest',
  'Depressed',
  'SleepTrouble',
  'PhysActive',
  'Alcohol12PlusYr',
  'SmokeNow',
  'Smoke100',
  'Smoke100n',
  'Marijuana',
  'RegularMarij',
  'HardDrugs',
  'SexEver',
  'SameSex'],
 ['ID',
  'Age',
  'AgeMonths',
  'HHIncomeMid',
  'HomeRooms',
  'Pulse',
  'BPSysAve',
  'BPDiaAve',
  'BPSys1',
  'BPDia1',
  'BPSys2',
  'BPDia2',
  'BPSys3',
  'BPDia3',
  'UrineVol1',
  'UrineVol2',
  'DiabetesAge',
  'DaysPhysHlthBad',
  'DaysMentHlthBad',
  'nPregnancies',
  'nBabies',
  'Age1stBaby',
  'SleepHrsNight',
  'PhysActiveDays',
  'TVHrsDayChild',
  'CompHrsDayChild',
  'AlcoholDay',
  'AlcoholYear',
  'SmokeAge',
  'AgeFirstMarij',
  'AgeRegMarij',
  'SexAge',
  'SexNumPartnLife',
  'SexNumPartYear'],
 ['ID',
  'Age',
  'AgeMonths',
  'HHIncomeMid',
  'Poverty',
  'HomeRooms',
  'Weight',
  'Length',
  'HeadCirc',
  'Height'])

In [8]:
data["Diabetes"].value_counts(dropna=False)   # example
data["Depressed"].value_counts(dropna=False)  # example

Depressed
NaN        8573
Several    1009
Most        418
Name: count, dtype: int64

In [9]:
data[["Age","BMI","Weight","Height","TotChol"]].describe()

,Age,BMI,Weight,Height,TotChol
count,10000.000000,9634.000000,9922.000000,9647.000000,8474.000000
mean,36.742100,26.660136,70.981798,161.877838,4.879220
std,22.397566,7.376579,29.125357,20.186567,1.075583
min,0.000000,12.880000,2.800000,83.600000,1.530000
25%,17.000000,21.580000,56.100000,156.800000,4.110000
50%,36.000000,25.980000,72.700000,166.000000,4.780000
75%,54.000000,30.890000,88.900000,174.500000,5.530000
max,80.000000,81.250000,230.700000,200.400000,13.650000


In [10]:
count_cols
data[count_cols].describe().T
data[count_cols[0]].value_counts().head(15)

ID
70324    8
69626    7
63297    7
62927    7
68499    6
67312    6
64675    6
63330    6
70653    6
63390    6
63744    6
66113    6
64850    6
68035    6
63163    6
Name: count, dtype: int64

Linear regression works when Y has a lot of unique values that are numbers (BMI, Weight, TotChol…).

Logistic works when Y is binary (answer choices are 0/1 or Yes/No) (Diabetes, Depressed, Smoke100…).

Poisson works when Y is a nonnegative count (integer, lots of 0’s, max > 5).

## **[2]**

In [11]:
cols = ["BMI", "Age", "Weight", "Height", "TotChol"]
df = data[cols].dropna().copy()
df.shape

(8406, 5)

In [12]:
X_cols = ["Age","Weight","Height","TotChol"]
corr = df[X_cols].corr()
corr

,Age,Weight,Height,TotChol
Age,1.000000,0.308399,0.251475,0.331181
Weight,0.308399,1.000000,0.637775,0.139527
Height,0.251475,0.637775,1.000000,0.090889
TotChol,0.331181,0.139527,0.090889,1.000000


In [13]:
pairs = []
for i in range(len(X_cols)):
    for j in range(i+1, len(X_cols)):
        r = corr.iloc[i,j]
        if abs(r) > 0.25:
            pairs.append((X_cols[i], X_cols[j], r))
pairs

[('Age', 'Weight', np.float64(0.3083989377528594)),
 ('Age', 'Height', np.float64(0.2514749907852697)),
 ('Age', 'TotChol', np.float64(0.3311809120600747)),
 ('Weight', 'Height', np.float64(0.6377748266876196))]

In [14]:
for a,b,r in pairs:
    df[f"{a}_x_{b}"] = df[a] * df[b]

df = df.reset_index(drop=True)
xs = X_cols + [f"{a}_x_{b}" for a,b,_ in pairs]
ys = ["BMI"]
train, test = partition_by_key(df, xs, ys)

Lin = LinearRegression().fit(df.loc[train][xs], df.loc[train][ys])
pred = Lin.predict(df.loc[test][xs])

rmse = float(np.sqrt(np.mean((df.loc[test]["BMI"].values - pred.ravel())**2)))
rmse

0.3306721991868521

In [15]:
y_corr = df[X_cols].corrwith(df["BMI"]).abs().sort_values(ascending=False)
y_corr

Weight     0.898479
Age        0.294697
Height     0.268957
TotChol    0.147741
dtype: float64

In [16]:
xs_min = y_corr.index[:2].tolist()
train, test = partition_by_key(df, xs_min, ["BMI"])
Lin_min = LinearRegression().fit(df.loc[train][xs_min], df.loc[train][["BMI"]])
pred_min = Lin_min.predict(df.loc[test][xs_min])

rmse_min = float(np.sqrt(np.mean((df.loc[test]["BMI"].values - pred_min.ravel())**2)))
rmse, rmse_min

(0.3306721991868521, 3.1996075773572663)

Linear Regression is good for BMI as it has a wide range of unique values

## **[3]**

In [17]:
cols = ["Diabetes","Age","BMI","TotChol","Weight"]
df = data[cols].dropna().copy()

# If Diabetes is Yes/No, convert to 0/1:
if df["Diabetes"].dtype == "object":
    df["Diabetes"] = df["Diabetes"].map({"No":0, "Yes":1})

df["Diabetes"].value_counts()

Diabetes
0    7712
1     689
Name: count, dtype: int64

In [18]:
cols = ["Diabetes","Age","BMI","TotChol","Weight"]
df = data[cols].dropna().copy()

# convert Diabetes to 0/1 if needed
if df["Diabetes"].dtype == "object":
    df["Diabetes"] = df["Diabetes"].map({"No":0, "Yes":1})

# reset index so partition_by_key + .loc work
df = df.reset_index(drop=True)

X_cols = ["Age","BMI","TotChol","Weight"]
corr = df[X_cols].corr()

pairs = []
for i in range(len(X_cols)):
    for j in range(i+1, len(X_cols)):
        r = corr.iloc[i,j]
        if abs(r) > 0.25:
            pairs.append((X_cols[i], X_cols[j], r))

for a,b,r in pairs:
    df[f"{a}_x_{b}"] = df[a] * df[b]

xs = X_cols + [f"{a}_x_{b}" for a,b,_ in pairs]
ys = ["Diabetes"]

train, test = partition_by_key(df, xs, ys)

Log = LogisticRegression(solver="liblinear").fit(df.loc[train][xs], df.loc[train][ys[0]])
probs = Log.predict_proba(df.loc[test][xs])[:,1]
pred = (probs >= 0.5).astype(int)

acc = float((pred == df.loc[test][ys[0]].values).mean())
acc

0.9011904761904762

In [19]:
y_corr = df[X_cols].corrwith(df["Diabetes"]).abs().sort_values(ascending=False)
y_corr

Age        0.268767
BMI        0.213707
Weight     0.181541
TotChol    0.025839
dtype: float64

Diabetes is binary because the answer choices are yes or no, so logistic regression is best here.

## **[4]**

In [39]:
import pandas as pd
import numpy as np

data = pd.read_csv("NHANES.csv")

Ycol = "DaysPhysHlthBad"
X_cols = ["Age", "BMI", "Weight", "Height"]

df = data[[Ycol] + X_cols].dropna().copy()

# correlations among predictors (to decide interactions)
corr = df[X_cols].corr()
corr

,Age,BMI,Weight,Height
Age,1.000000,0.183381,0.129891,-0.063956
BMI,0.183381,1.000000,0.889565,0.026774
Weight,0.129891,0.889565,1.000000,0.467381
Height,-0.063956,0.026774,0.467381,1.000000


In [40]:
# add interaction terms for pairs with |r| > 0.25
pairs = []
for i in range(len(X_cols)):
    for j in range(i+1, len(X_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.25:
            a, b = X_cols[i], X_cols[j]
            pairs.append((a, b, r))
            df[f"{a}_x_{b}"] = df[a] * df[b]

pairs

[('BMI', 'Weight', np.float64(0.8895650371513102)),
 ('Weight', 'Height', np.float64(0.4673806899659604))]

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import TweedieRegressor
from sklearn.metrics import mean_poisson_deviance, mean_absolute_error

feature_cols = X_cols + [f"{a}_x_{b}" for a, b, r in pairs]

X = df[feature_cols]
y = df[Ycol].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

glm = TweedieRegressor(power=1, alpha=0.5, link="log", max_iter=5000)
glm.fit(X_train, y_train)
pred = glm.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("Mean Poisson Deviance:", mean_poisson_deviance(y_test, pred))
print("R^2:", glm.score(X_test, y_test))

MAE: 4.7014369630137045
Mean Poisson Deviance: 10.336386272694664
R^2: -0.0003951334052112454


DaysPhysHealthBad has a lot of zeros, it's nonnegative, max > 5, so poisson is best here.